In [1]:
"""
════════════════════════════════════════════════════════════════════════════════════
  SAM2 vs MedSAM: Segmentation Backbone Comparison for WILLIE
  Notebook: 12_SAM2_vs_MedSAM_Comparison.ipynb
════════════════════════════════════════════════════════════════════════════════════

  PURPOSE: Justify the selection of SAM2.1 (Hiera-S) over MedSAM (ViT-B) as
           the segmentation backbone in the WILLIE multi-task architecture.

  DATA: Loads pre-computed results from seg_benchmark_master.pt
        (per-image Dice scores for U-Net, MedSAM, SAM2 on FUSeg val set)

  PRODUCES:
    1. fig_dice_barplot.png          — Mean + Median Dice comparison
    2. fig_dice_distribution.png     — Violin + box plots (per-image Dice)
    3. fig_head_to_head.png          — SAM2 vs MedSAM scatter (per-image)
    4. fig_quality_tiers.png         — % images at quality thresholds
    5. fig_efficiency.png            — Performance vs parameters tradeoff
    6. fig_statistical_tests.png     — Wilcoxon + paired difference histogram
    7. tbl_comparison_summary.png    — Publication-ready summary table

  All figures: 300 DPI, PNG + PDF, white background
════════════════════════════════════════════════════════════════════════════════════
"""

import os, sys, warnings
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

PROJECT_ROOT = "."
SEG_BENCH_DIR = os.path.join(PROJECT_ROOT, "artifacts/woundshot_v2/seg_benchmark")
MASTER_PATH = os.path.join(SEG_BENCH_DIR, "seg_benchmark_master.pt")

FIGURES_DIR = os.path.join(PROJECT_ROOT, "artifacts/12_sam2_vs_medsam/figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"{'='*80}")
print(f"  SAM2 vs MedSAM — Segmentation Backbone Comparison")
print(f"{'='*80}")
print(f"  📁 Figures: {FIGURES_DIR}")

# ── Load benchmark data ──
assert os.path.exists(MASTER_PATH), f"Run seg benchmark first! {MASTER_PATH} not found"
master = torch.load(MASTER_PATH, map_location="cpu", weights_only=False)
print(f"  📦 Loaded: {MASTER_PATH}")
print(f"  Keys: {list(master.keys())}")

# Extract per-model results
unet_r = master["unet"]
medsam_r = master["medsam"]
sam2_r = master["sam2"]

# Per-image dice scores
unet_dices = np.array(unet_r["dice_scores"])
medsam_dices = np.array(medsam_r["dice_scores"])
sam2_dices = np.array(sam2_r["dice_scores"])

N_IMAGES = len(sam2_dices)

print(f"\n  Models:")
print(f"    U-Net (Eff-B3):    {unet_r['dice_mean']:.4f} mean | {unet_r['dice_median']:.4f} median | {13.2}M params")
print(f"    MedSAM (ViT-B):    {medsam_r['dice_mean']:.4f} mean | {medsam_r['dice_median']:.4f} median | {93.7}M params")
print(f"    SAM2.1 (Hiera-S):  {sam2_r['dice_mean']:.4f} mean | {sam2_r['dice_median']:.4f} median | {46.1}M params")
print(f"    Images: {N_IMAGES}")

# ── Style ──
COLORS = {
    'unet':   '#95a5a6',  # gray - weakest
    'medsam': '#e74c3c',  # red
    'sam2':   '#2ecc71',  # green - winner
}
MODEL_NAMES = ['U-Net\n(Eff-B3)', 'MedSAM\n(ViT-B)', 'SAM2.1\n(Hiera-S)']
MODEL_SHORT = ['U-Net', 'MedSAM', 'SAM2']

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

def save_fig(fig, name, close=True):
    png = os.path.join(FIGURES_DIR, f"{name}.png")
    pdf = os.path.join(FIGURES_DIR, f"{name}.pdf")
    fig.savefig(png, dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    if close:
        plt.close(fig)
    print(f"  📈 {name} (.png + .pdf)")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: MEAN + MEDIAN DICE BAR CHART
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print(f"  FIGURE 1: Dice Score Comparison (Bar Chart)")
print(f"{'='*70}")

fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

means = [unet_r['dice_mean'], medsam_r['dice_mean'], sam2_r['dice_mean']]
medians = [unet_r['dice_median'], medsam_r['dice_median'], sam2_r['dice_median']]
stds = [unet_r['dice_std'], medsam_r['dice_std'], sam2_r['dice_std']]
colors = [COLORS['unet'], COLORS['medsam'], COLORS['sam2']]

# Mean Dice
bars1 = ax1.bar(range(3), means, yerr=stds, color=colors, edgecolor='black',
                linewidth=0.5, capsize=5, alpha=0.85, width=0.6)
for bar, val, std in zip(bars1, means, stds):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.008,
             f'{val:.2%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax1.set_xticks(range(3))
ax1.set_xticklabels(MODEL_NAMES, fontweight='bold')
ax1.set_ylabel('Mean Dice Score', fontweight='bold')
ax1.set_title('Mean Dice (± std)', fontweight='bold')
ax1.set_ylim([0.75, 1.0])
# Star for winner
ax1.annotate('★ Winner', xy=(2, means[2] + stds[2] + 0.025),
             ha='center', fontsize=10, color='#27ae60', fontweight='bold')

# Median Dice
bars2 = ax2.bar(range(3), medians, color=colors, edgecolor='black',
                linewidth=0.5, alpha=0.85, width=0.6)
for bar, val in zip(bars2, medians):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.2%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax2.set_xticks(range(3))
ax2.set_xticklabels(MODEL_NAMES, fontweight='bold')
ax2.set_ylabel('Median Dice Score', fontweight='bold')
ax2.set_title('Median Dice', fontweight='bold')
ax2.set_ylim([0.88, 0.96])

fig1.suptitle('Wound Segmentation Baseline Comparison — FUSeg Dataset (400 images)',
              fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
save_fig(fig1, "fig_dice_barplot")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: VIOLIN + BOX PLOTS (PER-IMAGE DISTRIBUTION)
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print(f"  FIGURE 2: Per-Image Dice Distribution")
print(f"{'='*70}")

fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [2, 1]})

# Left: Violin + strip
all_dices = [unet_dices, medsam_dices, sam2_dices]
parts = ax1.violinplot(all_dices, positions=[1, 2, 3], showmeans=True,
                       showmedians=True, widths=0.7)

for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.5)
parts['cmeans'].set_color('black')
parts['cmedians'].set_color('navy')

# Jittered scatter overlay
np.random.seed(42)
for i, (dices, pos) in enumerate(zip(all_dices, [1, 2, 3])):
    jitter = np.random.normal(pos, 0.06, len(dices))
    ax1.scatter(jitter, dices, c=colors[i], alpha=0.08, s=8, zorder=2)

ax1.set_xticks([1, 2, 3])
ax1.set_xticklabels(MODEL_NAMES, fontweight='bold')
ax1.set_ylabel('Per-Image Dice Score', fontweight='bold')
ax1.set_title('Per-Image Dice Score Distribution (400 images)', fontweight='bold')
ax1.axhline(0.9, color='green', linestyle='--', alpha=0.4, label='Dice=0.9')
ax1.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='Dice=0.5')
ax1.legend(fontsize=8, loc='lower left')

# Right: Overlaid histograms
bins = np.linspace(0, 1, 51)
ax2.hist(unet_dices, bins=bins, alpha=0.4, color=COLORS['unet'], label='U-Net', density=True)
ax2.hist(medsam_dices, bins=bins, alpha=0.5, color=COLORS['medsam'], label='MedSAM', density=True)
ax2.hist(sam2_dices, bins=bins, alpha=0.5, color=COLORS['sam2'], label='SAM2', density=True)
ax2.set_xlabel('Dice Score', fontweight='bold')
ax2.set_ylabel('Density', fontweight='bold')
ax2.set_title('Overlaid Distributions', fontweight='bold')
ax2.legend(fontsize=9)

fig2.suptitle('Per-Image Segmentation Quality — SAM2 vs MedSAM vs U-Net',
              fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
save_fig(fig2, "fig_dice_distribution")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: HEAD-TO-HEAD SAM2 vs MedSAM SCATTER
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print(f"  FIGURE 3: SAM2 vs MedSAM Head-to-Head")
print(f"{'='*70}")

fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

# Left: Scatter plot
n_match = min(len(sam2_dices), len(medsam_dices))
s2 = sam2_dices[:n_match]
ms = medsam_dices[:n_match]

# Color by which model wins
sam2_wins = s2 > ms
medsam_wins = ms > s2
ties = s2 == ms

ax1.scatter(ms[sam2_wins], s2[sam2_wins], c=COLORS['sam2'], alpha=0.4, s=20,
            label=f'SAM2 better ({sam2_wins.sum()}, {sam2_wins.mean():.0%})')
ax1.scatter(ms[medsam_wins], s2[medsam_wins], c=COLORS['medsam'], alpha=0.4, s=20,
            label=f'MedSAM better ({medsam_wins.sum()}, {medsam_wins.mean():.0%})')
ax1.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Equal performance')

ax1.set_xlabel('MedSAM Dice Score', fontweight='bold')
ax1.set_ylabel('SAM2 Dice Score', fontweight='bold')
ax1.set_title('Per-Image Head-to-Head Comparison', fontweight='bold')
ax1.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax1.set_xlim([0, 1.02])
ax1.set_ylim([0, 1.02])

# Annotate
n_sam2_better = sam2_wins.sum()
n_medsam_better = medsam_wins.sum()
ax1.text(0.02, 0.98, f'SAM2 wins on {n_sam2_better}/{n_match} images\n'
         f'MedSAM wins on {n_medsam_better}/{n_match} images',
         transform=ax1.transAxes, ha='left', va='top', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray', alpha=0.9))

# Right: Paired difference histogram
diff = s2 - ms
ax2.hist(diff, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
ax2.axvline(0, color='black', linestyle='-', lw=2)
ax2.axvline(np.mean(diff), color='green', linestyle='--', lw=2,
            label=f'Mean diff: {np.mean(diff):+.4f}')
ax2.axvline(np.median(diff), color='navy', linestyle=':', lw=2,
            label=f'Median diff: {np.median(diff):+.4f}')
ax2.set_xlabel('SAM2 Dice − MedSAM Dice', fontweight='bold')
ax2.set_ylabel('Count', fontweight='bold')
ax2.set_title('Paired Difference Distribution', fontweight='bold')
ax2.legend(fontsize=9)

# Shade positive region
ax2.axvspan(0, max(diff), alpha=0.05, color='green')
ax2.text(0.02, 0.95, 'SAM2\nbetter →', transform=ax2.transAxes,
         fontsize=8, color='green', va='top')
ax2.text(0.98, 0.95, '← MedSAM\nbetter', transform=ax2.transAxes,
         fontsize=8, color='red', va='top', ha='right')

fig3.suptitle('SAM2 vs MedSAM — Per-Image Paired Comparison (400 images)',
              fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
save_fig(fig3, "fig_head_to_head")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 4: QUALITY TIER COMPARISON
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print(f"  FIGURE 4: Quality Tiers")
print(f"{'='*70}")

fig4, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

thresholds = [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]
unet_pcts = [np.mean(unet_dices >= t) * 100 for t in thresholds]
medsam_pcts = [np.mean(medsam_dices >= t) * 100 for t in thresholds]
sam2_pcts = [np.mean(sam2_dices >= t) * 100 for t in thresholds]

# Left: Line plot
ax1.plot(thresholds, unet_pcts, 'o-', color=COLORS['unet'], lw=2, markersize=6, label='U-Net')
ax1.plot(thresholds, medsam_pcts, 's-', color=COLORS['medsam'], lw=2, markersize=6, label='MedSAM')
ax1.plot(thresholds, sam2_pcts, 'D-', color=COLORS['sam2'], lw=2, markersize=6, label='SAM2')
ax1.set_xlabel('Dice Threshold', fontweight='bold')
ax1.set_ylabel('% Images Above Threshold', fontweight='bold')
ax1.set_title('Cumulative Quality Profile', fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_ylim([0, 105])

# Right: Grouped bars at key thresholds
key_thresh = [0.8, 0.9, 0.95]
x = np.arange(len(key_thresh))
w = 0.25
u_vals = [np.mean(unet_dices >= t) * 100 for t in key_thresh]
m_vals = [np.mean(medsam_dices >= t) * 100 for t in key_thresh]
s_vals = [np.mean(sam2_dices >= t) * 100 for t in key_thresh]

b1 = ax2.bar(x - w, u_vals, w, color=COLORS['unet'], label='U-Net', edgecolor='white')
b2 = ax2.bar(x, m_vals, w, color=COLORS['medsam'], label='MedSAM', edgecolor='white')
b3 = ax2.bar(x + w, s_vals, w, color=COLORS['sam2'], label='SAM2', edgecolor='white')

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2, h + 0.8, f'{h:.0f}%',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels([f'Dice ≥ {t}' for t in key_thresh], fontweight='bold')
ax2.set_ylabel('% of Images', fontweight='bold')
ax2.set_title('Quality Tier Breakdown', fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_ylim([0, 110])

fig4.suptitle('Segmentation Quality Tiers — Higher is Better',
              fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
save_fig(fig4, "fig_quality_tiers")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 5: EFFICIENCY COMPARISON
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print(f"  FIGURE 5: Efficiency (Performance vs Parameters)")
print(f"{'='*70}")

fig5, ax = plt.subplots(figsize=(8, 6))

params = [13.2, 93.7, 46.1]
trainable = [13.2, 4.06, 11.74]
mean_dice = [unet_r['dice_mean'] * 100, medsam_r['dice_mean'] * 100, sam2_r['dice_mean'] * 100]
train_times = [15.5, 36.6, 53.6]  # minutes

# Bubble size = training time
sizes = [t * 8 for t in train_times]
scatter = ax.scatter(params, mean_dice, s=sizes, c=colors, alpha=0.7,
                     edgecolors='black', linewidths=1.5, zorder=3)

# Labels
for i, (name, p, d) in enumerate(zip(MODEL_SHORT, params, mean_dice)):
    ax.annotate(f'{name}\n{d:.1f}% | {p}M params\n{train_times[i]:.0f}min',
                xy=(p, d), xytext=(p + 5, d + 0.3),
                fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='gray', alpha=0.5),
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

ax.set_xlabel('Total Parameters (M)', fontweight='bold', fontsize=12)
ax.set_ylabel('Mean Dice Score (%)', fontweight='bold', fontsize=12)
ax.set_title('Performance vs Model Size\n(Bubble size ∝ training time)',
             fontweight='bold', fontsize=14, pad=15)

# Add annotation box
ax.text(0.98, 0.05,
        'SAM2 achieves highest Dice\nwith 2× fewer params than MedSAM\n'
        'and much higher Dice than U-Net',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#d5f5e3', edgecolor='#27ae60', alpha=0.9))

save_fig(fig5, "fig_efficiency")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 6: STATISTICAL SIGNIFICANCE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print(f"  FIGURE 6: Statistical Tests")
print(f"{'='*70}")

fig6, axes = plt.subplots(1, 3, figsize=(16, 5))

# Wilcoxon signed-rank test: SAM2 vs MedSAM
stat_sm, p_sm = stats.wilcoxon(s2, ms, alternative='greater')
# SAM2 vs U-Net
n_su = min(len(sam2_dices), len(unet_dices))
stat_su, p_su = stats.wilcoxon(sam2_dices[:n_su], unet_dices[:n_su], alternative='greater')
# MedSAM vs U-Net
n_mu = min(len(medsam_dices), len(unet_dices))
stat_mu, p_mu = stats.wilcoxon(medsam_dices[:n_mu], unet_dices[:n_mu], alternative='greater')

print(f"  Wilcoxon signed-rank (one-sided, H1: model_A > model_B):")
print(f"    SAM2 > MedSAM:  W={stat_sm:.1f}, p={p_sm:.2e}  {'★ significant' if p_sm < 0.05 else 'not sig.'}")
print(f"    SAM2 > U-Net:   W={stat_su:.1f}, p={p_su:.2e}  {'★ significant' if p_su < 0.05 else 'not sig.'}")
print(f"    MedSAM > U-Net: W={stat_mu:.1f}, p={p_mu:.2e}  {'★ significant' if p_mu < 0.05 else 'not sig.'}")

# Panel 1: Paired difference SAM2 - MedSAM
diff_sm = s2 - ms
axes[0].hist(diff_sm, bins=40, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='black', lw=2)
axes[0].axvline(np.mean(diff_sm), color='green', linestyle='--', lw=2,
                label=f'Mean: {np.mean(diff_sm):+.4f}')
axes[0].set_xlabel('SAM2 − MedSAM (Dice)')
axes[0].set_ylabel('Count')
sig_str = f'p={p_sm:.2e} ★' if p_sm < 0.05 else f'p={p_sm:.3f}'
axes[0].set_title(f'SAM2 vs MedSAM\nWilcoxon: {sig_str}', fontweight='bold')
axes[0].legend(fontsize=9)

# Panel 2: Paired difference SAM2 - U-Net
diff_su = sam2_dices[:n_su] - unet_dices[:n_su]
axes[1].hist(diff_su, bins=40, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='black', lw=2)
axes[1].axvline(np.mean(diff_su), color='green', linestyle='--', lw=2,
                label=f'Mean: {np.mean(diff_su):+.4f}')
axes[1].set_xlabel('SAM2 − U-Net (Dice)')
axes[1].set_ylabel('Count')
sig_str2 = f'p={p_su:.2e} ★' if p_su < 0.05 else f'p={p_su:.3f}'
axes[1].set_title(f'SAM2 vs U-Net\nWilcoxon: {sig_str2}', fontweight='bold')
axes[1].legend(fontsize=9)

# Panel 3: Summary table of stats
axes[2].axis('off')
stat_table = [
    ['Comparison', 'Mean Δ', 'W-stat', 'p-value', 'Sig?'],
    ['SAM2 > MedSAM', f'{np.mean(diff_sm):+.4f}', f'{stat_sm:.0f}', f'{p_sm:.2e}',
     '★★★' if p_sm < 0.001 else ('★★' if p_sm < 0.01 else ('★' if p_sm < 0.05 else 'No'))],
    ['SAM2 > U-Net', f'{np.mean(diff_su):+.4f}', f'{stat_su:.0f}', f'{p_su:.2e}',
     '★★★' if p_su < 0.001 else ('★★' if p_su < 0.01 else ('★' if p_su < 0.05 else 'No'))],
    ['MedSAM > U-Net', f'{np.mean(medsam_dices[:n_mu] - unet_dices[:n_mu]):+.4f}',
     f'{stat_mu:.0f}', f'{p_mu:.2e}',
     '★★★' if p_mu < 0.001 else ('★★' if p_mu < 0.01 else ('★' if p_mu < 0.05 else 'No'))],
]

tbl = axes[2].table(cellText=stat_table[1:], colLabels=stat_table[0],
                    cellLoc='center', loc='center', colWidths=[0.22, 0.15, 0.15, 0.22, 0.1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)
for j in range(5):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, 4):
    for j in range(5):
        tbl[i, j].set_facecolor('#ecf0f1' if i % 2 == 0 else 'white')
axes[2].set_title('Wilcoxon Signed-Rank Tests\n(one-sided)', fontweight='bold')

fig6.suptitle('Statistical Significance — Paired Per-Image Comparison',
              fontweight='bold', fontsize=14, y=1.03)
plt.tight_layout()
save_fig(fig6, "fig_statistical_tests")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 7: PUBLICATION SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════════

print(f"\n{'='*70}")
print(f"  FIGURE 7: Summary Table")
print(f"{'='*70}")

fig7, ax = plt.subplots(figsize=(15, 5))
ax.axis('off')

pct_90 = [np.mean(d >= 0.9) * 100 for d in [unet_dices, medsam_dices, sam2_dices]]

table_data = [
    ['Model', 'Backbone', 'Total\nParams', 'Trainable\nParams', 'Mean\nDice',
     'Median\nDice', 'Std', 'Dice≥0.9', 'Train\nTime', 'Reference'],
    ['U-Net', 'EfficientNet-B3', '13.2M', '13.2M',
     f'{unet_r["dice_mean"]:.2%}', f'{unet_r["dice_median"]:.2%}',
     f'{unet_r["dice_std"]:.4f}', f'{pct_90[0]:.0f}%', '15.5 min',
     'Ronneberger (2015)'],
    ['MedSAM', 'ViT-B', '93.7M', '4.06M',
     f'{medsam_r["dice_mean"]:.2%}', f'{medsam_r["dice_median"]:.2%}',
     f'{medsam_r["dice_std"]:.4f}', f'{pct_90[1]:.0f}%', '36.6 min',
     'Ma et al. (2024)'],
    ['SAM2.1 ★', 'Hiera-S', '46.1M', '11.74M',
     f'{sam2_r["dice_mean"]:.2%}', f'{sam2_r["dice_median"]:.2%}',
     f'{sam2_r["dice_std"]:.4f}', f'{pct_90[2]:.0f}%', '53.6 min',
     'Ravi et al. (2024)'],
]

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                 cellLoc='center', loc='center',
                 colWidths=[0.09, 0.1, 0.07, 0.08, 0.07, 0.07, 0.06, 0.07, 0.07, 0.13])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)

# Header
for j in range(len(table_data[0])):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Row colors
row_colors = ['#ffffff', '#ffffff', '#d5f5e3']  # highlight SAM2 row green
for i in range(1, 4):
    for j in range(len(table_data[0])):
        table[i, j].set_facecolor(row_colors[i-1])
# Bold SAM2 row
for j in range(len(table_data[0])):
    table[3, j].set_text_props(fontweight='bold')

ax.set_title('Segmentation Baseline Comparison — FUSeg Dataset (610 train / 400 val)\n'
             'All models fine-tuned with frozen encoder, bbox prompt, BCE+Dice loss',
             fontweight='bold', fontsize=13, pad=20)

save_fig(fig7, "tbl_comparison_summary")


# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

fig_files = sorted([f for f in os.listdir(FIGURES_DIR) if f.endswith(('.png', '.pdf'))])

print(f"""
{'='*80}
  ✅ COMPARISON COMPLETE — SAM2 vs MedSAM
{'='*80}

  KEY FINDINGS:
    • SAM2.1 (Hiera-S) achieves HIGHEST Dice: {sam2_r['dice_mean']:.2%} mean
    • SAM2 uses 2× FEWER params than MedSAM (46.1M vs 93.7M)
    • SAM2 beats MedSAM on {n_sam2_better}/{n_match} individual images ({n_sam2_better/n_match:.0%})
    • Wilcoxon: SAM2 > MedSAM is {'statistically significant' if p_sm < 0.05 else 'not significant'} (p={p_sm:.2e})
    • Both SAM variants dramatically outperform U-Net (+6-7% mean Dice)

  JUSTIFICATION FOR WILLIE:
    SAM2.1 selected as segmentation backbone because:
    1. Highest mean Dice (91.74% vs 90.93%)
    2. Highest median Dice (93.76% vs 93.33%)
    3. Better per-image win rate ({n_sam2_better/n_match:.0%} of images)
    4. Fewer parameters (46.1M vs 93.7M) — more efficient
    5. Hiera architecture enables multi-scale features for FPN fusion
    6. Native support for video/temporal — future clinical deployment

  📁 {FIGURES_DIR}
  📊 {len(fig_files)} files generated
""")

for f in fig_files:
    size = os.path.getsize(os.path.join(FIGURES_DIR, f))
    print(f"     {'📈' if f.endswith('.png') else '📄'} {f}  ({size/1024:.1f} KB)")

print(f"""
  ┌───────────────────────────────────────────────────────────┐
  │  FIGURE INVENTORY                                         │
  ├───────────────────────────────────────────────────────────┤
  │  1. fig_dice_barplot         — Mean + Median bars          │
  │  2. fig_dice_distribution    — Violin + histogram overlay  │
  │  3. fig_head_to_head         — SAM2 vs MedSAM scatter      │
  │  4. fig_quality_tiers        — % images above thresholds   │
  │  5. fig_efficiency           — Dice vs params bubble chart  │
  │  6. fig_statistical_tests    — Wilcoxon + diff histograms  │
  │  7. tbl_comparison_summary   — Publication results table   │
  └───────────────────────────────────────────────────────────┘
""")

  SAM2 vs MedSAM — Segmentation Backbone Comparison
  📁 Figures: artifacts/12_sam2_vs_medsam/figures
  📦 Loaded: artifacts/woundshot_v2/seg_benchmark/seg_benchmark_master.pt
  Keys: ['benchmark_df', 'winner', 'winner_dice', 'unet', 'medsam', 'sam2']

  Models:
    U-Net (Eff-B3):    0.8459 mean | 0.9120 median | 13.2M params
    MedSAM (ViT-B):    0.9093 mean | 0.9333 median | 93.7M params
    SAM2.1 (Hiera-S):  0.9174 mean | 0.9376 median | 46.1M params
    Images: 400

  FIGURE 1: Dice Score Comparison (Bar Chart)
  📈 fig_dice_barplot (.png + .pdf)

  FIGURE 2: Per-Image Dice Distribution
  📈 fig_dice_distribution (.png + .pdf)

  FIGURE 3: SAM2 vs MedSAM Head-to-Head
  📈 fig_head_to_head (.png + .pdf)

  FIGURE 4: Quality Tiers
  📈 fig_quality_tiers (.png + .pdf)

  FIGURE 5: Efficiency (Performance vs Parameters)
  📈 fig_efficiency (.png + .pdf)

  FIGURE 6: Statistical Tests
  Wilcoxon signed-rank (one-sided, H1: model_A > model_B):
    SAM2 > MedSAM:  W=44891.0, p=2.91e-04  ★ sig